# One-dimensional Fermi–Hubbard dynamics

All 39 source stages: four-mode factorization, four-site eight-mode Slater preparation, quarter-filling interacting dynamics on steps 0–9, one-electron noninteracting dynamics and half-filling vanishing hopping on steps 0–10, and two 400-point classical one-electron comparisons.

Fresh real eigendecomposition and adjacent row rotations replace the OpenFermion factorization for these number-conserving real matrices. The occupied Slater determinant is checked independently from its minors. Native rotations and Pauli exchange preserve the source little-endian mode order, the implemented positive contact phase and the five Trotter stages. Each time point binds a static step count; exact named-state occupation expectations replace hosted finite-shot observe calls. The spread plot retains the final source formula with center (N + 1)/2. Exact expectations do not reproduce the original shot noise or hosted trajectory.

## Use the existing FlagQuantum environment

In [ ]:
from pathlib import Path

import torch
from IPython.display import Image, display


## Import the native model and workflows

In [ ]:
from examples.synthesis.applications.physical_systems.fermi_hubbard_model_1d.fermi_hubbard_1d.overview import (
    model,
    plots,
    quantum,
    workflow,
)

artifact_dir = Path("artifacts/development/fermi-hubbard-1d")


## Keep the original system sizes

In [ ]:
sites, modes, interacting_points = 4, 8, 10
print("Sites, modes and interacting time points:", sites, modes, interacting_points)


## Generate the four-mode demonstration Hamiltonian

In [ ]:
example = model.example_matrix()
print(example)


## Diagonalize the demonstration matrix

In [ ]:
demo = workflow.demo_orbitals()
print("One-body energies:", demo.energies.tolist())


## Factor the occupied modes into adjacent rotations

In [ ]:
demo_plan = model.factor(demo.vectors)
print("Adjacent rotations:", demo_plan.rotations)


## Check the row-rotation inverse

In [ ]:
reconstructed = model.reconstruct_orbitals(demo_plan, 4)
print("Reconstructed occupied orbitals:", reconstructed)


## Check reconstruction against the eigenspace

In [ ]:
torch.testing.assert_close(reconstructed, demo.vectors, rtol=0, atol=1e-13)


## Check the single-excitation eigenvector

In [ ]:
one = workflow.diagonalize(example, 1)
torch.testing.assert_close(
    example @ one.vectors, one.vectors * one.energies[0], rtol=0, atol=1e-13
)


## Describe the source mode convention

In [ ]:
print("Spin-up modes 0..3; spin-down modes 4..7")


## Use source-sized native Slater preparation

In [ ]:
print("Adjacent Givens rotations and occupied-mode signs compose the Slater state.")


## Construct open-chain hopping and Gaussian potential

In [ ]:
potential = model.hamiltonian(3.0, 0.5)
print(potential)


## Select the source ground orbital

In [ ]:
single_orbital = workflow.diagonalize(potential, 1)
print("Ground energy:", float(single_orbital.energies[0]))


## Synthesize the single-electron program

In [ ]:
single_program = quantum.build_program(
    model.factor(single_orbital.vectors), 0, 0.1, 0.0
)
print(single_program.report["resources"])


## Read the full prepared state

In [ ]:
single, exact_amplitudes, fidelity = workflow.initial_one_electron()
print("State entries:", single.states.shape[-1])


## Extract single-particle amplitudes

In [ ]:
print(single.states[0][1 << torch.arange(8)])


## Compare against the exact ground orbital

In [ ]:
print("Ground-state fidelity:", fidelity)
assert fidelity > 0.95
display(
    Image(
        filename=str(
            plots.amplitudes(
                single, exact_amplitudes, artifact_dir / "initial-amplitudes.png"
            )
        )
    )
)


## Choose quarter filling

In [ ]:
occupied = 2
quarter = workflow.diagonalize(potential, occupied)


## Synthesize the quarter-filling state

In [ ]:
quarter_plan = model.factor(quarter.vectors)
print(quantum.build_program(quarter_plan, 0, 0.1, 0.0).report["resources"])


## Define charge and spin readouts

In [ ]:
print("Charge = n(up) + n(down); spin = n(up) − n(down).")


## Measure the initial densities

In [ ]:
initial = workflow.evolve(quarter.vectors, 1, 0.1, 1.0, 0.0)
print(initial.charge[0], initial.spin[0])


## Plot the initial densities

In [ ]:
display(
    Image(
        filename=str(
            plots.initial_densities(initial, artifact_dir / "initial-densities.png")
        )
    )
)


## Set the on-site interaction

In [ ]:
contact = 3.0
print("U/J =", contact)


## Describe the exchange and contact operations

In [ ]:
print(
    "Exchange uses XX and YY; the contact applies the source positive phase to double occupation."
)


## Keep hopping and contact site order

In [ ]:
print(
    "Odd-edge pairs (0, 1), (2, 3), in both spin sectors; odd-site contacts at 0 and 2."
)


## Check the middle-edge mode swap

In [ ]:
print("Swapped even-site contacts land at sites 2 and 3.")


## Compose the five stages

In [ ]:
print(
    "Odd hopping, odd contact, middle mode swap, even contact, undo with middle hopping."
)


## Synthesize the last interacting step

In [ ]:
print(quantum.build_program(quarter_plan, 9, 0.1, 0.3).report["resources"])


## Define the spread readout

In [ ]:
print("Use the final source spread weights with center (N + 1)/2.")


## Run all interacting time points

In [ ]:
interacting = workflow.interacting()
print("Interacting time points:", len(interacting.times))


## Show charge and spin snapshots

In [ ]:
display(
    Image(
        filename=str(
            plots.dynamics(interacting, artifact_dir / "interacting-densities.png")
        )
    )
)


## Show charge and spin spread

In [ ]:
display(
    Image(filename=str(plots.spread_plot(interacting, artifact_dir / "spread.png")))
)


## Set up the noninteracting one-electron limit

In [ ]:
print("Free case: U=0, J=1, ten steps, dt=0.1, localized spin-up electron.")


## Compare open-chain exact descriptions

In [ ]:
dispersion, numerical, analytic = workflow.one_particle_curves(
    torch.linspace(0, 4, 400, dtype=torch.float64)
)
print("Open-chain dispersion:", dispersion)
print("Analytic grid points:", len(analytic))
display(Image(filename=str(plots.free_dynamics(artifact_dir / "free-analytic.png"))))


## Run all free-chain steps

In [ ]:
free = workflow.noninteracting()
print("Noninteracting time points:", len(free.times))
display(
    Image(filename=str(plots.free_dynamics(artifact_dir / "free-trotter.png", free)))
)


## Set up the vanishing-hopping limit

In [ ]:
print("J=0, U=2, four occupied modes, ten steps, dt=0.3.")


## Run all vanishing-hopping steps

In [ ]:
frozen = workflow.vanishing_hopping()
print("Vanishing hopping time points:", len(frozen.times))


## Check frozen densities throughout the trajectory

In [ ]:
charge_error = float((frozen.charge - frozen.charge[0]).abs().max())
spin_error = float((frozen.spin - frozen.spin[0]).abs().max())
print("Frozen density errors:", charge_error, spin_error)
assert max(charge_error, spin_error) < 1e-12


## Compare final and initial frozen densities

In [ ]:
display(
    Image(
        filename=str(
            plots.frozen_densities(frozen, artifact_dir / "frozen-densities.png")
        )
    )
)
